In [0]:
import plotly.express as px
import pandas as pd
from pyspark.sql.functions import col

# --- FUNÇÃO AUXILIAR: LIMPEZA DE TEXTO ---
def limpar_texto_ia(texto):
    if not texto: return "Diagnóstico indisponível."
    if len(texto) > 100 and texto.count(texto[:10]) > 2:
        return texto.split('.')[0] + "."
    frases = list(dict.fromkeys(texto.split('. '))) 
    return '. '.join(frases)

# --- 1. CARREGAR DADOS ---
print("🔄 Carregando dados de alertas...")
df_analise = spark.read.table("olist_portfolio.gold.ai_diagnostics").toPandas()
df_analise['diagnostico_ia'] = df_analise['diagnostico_ia'].astype(str).apply(limpar_texto_ia)


# --- 2. AUTOMAÇÃO EM MASSA ---
print("\n" + "═"*80)
print(f"🤖 DISPARO DE AÇÕES AUTOMÁTICAS ({len(df_analise)} Alertas)")
print("═"*80)

# Loop para gerar e-mail de TODAS as categorias críticas
for index, row in df_analise.iterrows():
    cat_nome = row['categoria'].upper()
    
    print(f"\n📍 PROCESSANDO: {cat_nome}")
    print(f"   📝 Diagnóstico IA: \"{row['diagnostico_ia'][:100]}...\"") 

    email_body = f"""
    ASSUNTO: URGENTE - SUSPENSÃO DE VENDAS: {cat_nome}

    Prezados,

    A categoria '{row['categoria']}' apresentou {row['taxa_reprovacao']:.1f}% de reprovação.
    
    ANÁLISE DE CAUSA RAIZ (IA):
    "{row['diagnostico_ia']}"

    Ação Tomada: Bloqueio imediato de fornecedores associados.
    """
    
    print(f"\n📧 E-MAIL ENVIADO PARA O GESTOR DE {cat_nome}:")
    print(email_body)
    print("-" * 80)